# Notebook 01 — Data Profiling & Quality Assessment

1. Objectives
- Understand the datasets
- Assess data quality
- Identify issues before cleaning
- Produce profiling statistics

2. Imports & Configuration

In [28]:
import pandas as pd

3. Load Datasets

In [29]:
transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\transactions_stage-V2.csv")
orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\orders_stage.csv")
customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\customers_stage-V2.csv")
catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\RawData\catalogue_stage_V3.csv")

In [30]:
datasets = {
    "Transactions": transactions,
    "Orders": orders,
    "Customers": customers,
    "Catalogue": catalogue,
}

4. Dataset Overview

-  Data shape, structure

In [32]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.shape)
    print(df.info())
    print(df.dtypes)
    


Transactions
(20405, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20405 entries, 0 to 20404
Data columns (total 23 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   order_id_stage                    20405 non-null  object 
 1   Code Client                       20405 non-null  object 
 2   order_date                        20405 non-null  object 
 3   wilaya_raw                        20405 non-null  object 
 4   wilaya_normalized                 20405 non-null  object 
 5   geo_quality_flag                  20405 non-null  object 
 6   customer_type_inferred            20405 non-null  object 
 7   sku                               20405 non-null  object 
 8   product_name                      20267 non-null  object 
 9   sku_quality                       20405 non-null  object 
 10  category                          20405 non-null  object 
 11  subcategory                       12895 n

5. Data profiling

- nulls


In [40]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Transactions
order_id_stage                         0
Code Client                            0
order_date                             0
wilaya_raw                             0
wilaya_normalized                      0
geo_quality_flag                       0
customer_type_inferred                 0
sku                                    0
product_name                         138
sku_quality                            0
category                               0
subcategory                         7510
quantity                             125
unit_price                           125
line_total                           125
order_status                           0
payment_method_group                   0
Montant total de la commande           0
Titre moyen du paiement               16
sales_channel                          0
Titre de la méthode d’expédition      84
Poids total                            0
Montant de l’expédition commande       0
dtype: int64

Orders
order_id_stage        

- Duplicates

In [41]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.duplicated().sum())
    print(df.nunique())


Transactions
12
order_id_stage                      9241
Code Client                         5775
order_date                          1531
wilaya_raw                            60
wilaya_normalized                     59
geo_quality_flag                       4
customer_type_inferred                 2
sku                                  732
product_name                        1351
sku_quality                            4
category                               4
subcategory                           43
quantity                             124
unit_price                          1787
line_total                          3621
order_status                           9
payment_method_group                   5
Montant total de la commande        4853
Titre moyen du paiement               17
sales_channel                          1
Titre de la méthode d’expédition     107
Poids total                         2392
Montant de l’expédition commande     128
dtype: int64

Orders
0
order_id_stage   

In [35]:
catalogue[catalogue.duplicated()]

,SkU,Nom,categorie,Sous-catégorie,Prix unitaire
12,SKU-27803,Baguette de vitre,Menuiserie,Portes > Accessoires Portes,450.0
79,SKU-33813,Cadre de porte,Menuiserie,Portes,15000.0
80,SKU-33813,Cadre de porte,Menuiserie,Portes,15000.0
81,SKU-33813,Cadre de porte,Menuiserie,Portes,15000.0
82,SKU-33813,Cadre de porte,Menuiserie,Portes,15000.0
...,...,...,...,...,...
581,SKU-33814,Ouvrant de porte,Menuiserie,Portes,30000.0
582,SKU-33814,Ouvrant de porte,Menuiserie,Portes,30000.0
583,SKU-33814,Ouvrant de porte,Menuiserie,Portes,30000.0
611,SKU-28789,Poignée avec clé,Menuiserie,Portes > Accessoires Portes,2500.0


In [36]:
catalogue[catalogue["SkU"].isnull()]

,SkU,Nom,categorie,Sous-catégorie,Prix unitaire


- Descriptive statistics

In [37]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.describe(include="all"))



Transactions
       order_id_stage  Code Client  order_date wilaya_raw wilaya_normalized  \
count           20405        20405       20405      20405             20405   
unique           9241         5775        1531         60                59   
top       CMD_S008870  CLT_S002151  2026-06-17      Alger             Alger   
freq               57          160         115       4656              4656   
mean              NaN          NaN         NaN        NaN               NaN   
std               NaN          NaN         NaN        NaN               NaN   
min               NaN          NaN         NaN        NaN               NaN   
25%               NaN          NaN         NaN        NaN               NaN   
50%               NaN          NaN         NaN        NaN               NaN   
75%               NaN          NaN         NaN        NaN               NaN   
max               NaN          NaN         NaN        NaN               NaN   

       geo_quality_flag customer_type

- Business Integrity Checks

- transactions

In [38]:
#negative quantity
negative_quantity = transactions[transactions["quantity"] < 0]
print(f"Negative quantities: {len(negative_quantity)}")
negative_quantity.head()
#negative price
negative_price = transactions[transactions["unit_price"]<0]
print(f"Negative prices: {len(negative_price)}")
negative_price.head()
#negative_line_totals
negative_total = transactions[transactions["line_total"] < 0]
print(f"Negative line totals: {len(negative_total)}")
negative_total.head()
#future order
transactions["order_date"] = pd.to_datetime(
    transactions["order_date"],
    errors="coerce"
)
future_date=transactions[transactions["order_date"] > pd.Timestamp.today()]
print(f"Future dates:{len(future_date)}")

Negative quantities: 0
Negative prices: 65
Negative line totals: 65
Future dates:0


## Key findings
- ### Dataset Quality
- All four datasets were successfully loaded and inspected.
- No missing values were detected in the Orders and Customers datasets.
- Most columns were imported as object and require data type standardization (dates, text, and categorical variables).
- ### Missing Values
- #### Transactions
- 138  missing product_name values.
- 7510  missing subcategory values.
- 125 missing values in quantity, unit_price, and line_total.
- #### Catalogue
- 218 missing unit_price values.
- 2819 missing SKU values
- 93 duplicates.
- ### Duplicate Records
- 12 duplicate rows were detected in the Transactions dataset.
(These records require further investigation to determine whether they are true duplicates or valid business records.)
- ### Business Rule Validation
- No negative quantities were found.
- 65 records contain negative unit_price values.
- The same 65 records also contain negative line_total values, suggesting they may represent refunds, returns, or accounting adjustments rather than data entry errors.
- ### Initial Observations
- The 125 transaction records with missing financial information appear to correspond to zero-value orders and will require a business decision before cleaning.
